# Exploring the lakehouse

The Delta tables the hourly DAGs write, read through `lakehouse.py` — which
supplies the explicit schema each table needs, since the merge deliberately
keeps every payload as a JSON string.

In the **Pipeline Notebook** add-on this runs as-is. Elsewhere, set
`SPARK_CONNECT_URL` or edit the next cell.


In [ ]:
import os, sys
sys.path.insert(0, "/share/pipeline-airflow/lib")

from pyspark.sql import SparkSession, functions as F, Window
from lakehouse import table, tables, day, total_reps, held_seconds

spark = SparkSession.builder.remote(
    os.environ.get("SPARK_CONNECT_URL", "sc://172.30.32.1:15002")
).getOrCreate()

gym = tables(spark, "gym_tracker")
sorted(gym)


## Two things that will mislead you if unstated

**1. `sets` is often null.** The app stores nothing for a single-set entry, so
`sets * reps` nulls those rows and whole days vanish from a total. Use
`total_reps()`, which mirrors the app's own `COALESCE(sets, 1)`.

**2. `_actor` is null for everything the bootstrap loaded.** A snapshot is
state, not a change, so it has no single author. Only rows that arrived as
*changes* carry `user` / `automation` / `migration`. That is a property of when
a row was loaded, not of the row.


In [ ]:
w = gym["workout_logs"]
print("rows:", w.count())
w.groupBy("_actor").count().orderBy("_actor").show()


## Training volume over time


In [ ]:
by_day = (w.withColumn("day", day("ts"))
           .withColumn("r", total_reps())
           .withColumn("s", held_seconds())
           .groupBy("day")
           .agg(F.sum("r").alias("reps"),
                F.sum("s").alias("held_sec"),
                F.count("*").alias("entries"),
                F.round(F.avg("hr_avg"), 1).alias("hr_avg"))
           .orderBy("day"))
by_day.toPandas()


### Which exercises carry the volume


In [ ]:
ex = gym["exercises"].select(F.col("id").alias("exercise_id"),
                             F.col("name").alias("exercise"),
                             "category", "measure")

(w.join(ex, "exercise_id", "left")
  .withColumn("r", total_reps())
  .withColumn("s", held_seconds())
  .groupBy("exercise", "measure")
  .agg(F.count("*").alias("sessions"),
       F.sum("r").alias("reps"),
       F.sum("s").alias("held_sec"),
       F.round(F.avg("hr_avg"), 1).alias("hr_avg"))
  .orderBy(F.col("sessions").desc())
  .toPandas())


## Challenge adherence

Completions joined back to their item and challenge. Note the join to
`challenge_items` uses the item id — an item can move between challenges, so
its *current* challenge is not necessarily the one it was ticked under.


In [ ]:
items = gym["challenge_items"].select(
    F.col("id").alias("item_id"), F.col("label"),
    F.col("challenge_id"), F.col("item_type"))
chal = gym["challenges"].select(
    F.col("id").alias("challenge_id"), F.col("name").alias("challenge"),
    "start_date", "end_date")

done = (gym["challenge_completions"]
        .join(items, "item_id", "left")
        .join(chal, "challenge_id", "left"))

(done.groupBy("challenge", "label")
     .agg(F.count("*").alias("days_done"),
          F.min("day").alias("first"), F.max("day").alias("last"))
     .orderBy(F.col("days_done").desc())
     .toPandas())


### What gets taken back

Un-ticking really deletes, and the feed reports it — so "logged then undone"
is answerable, which a plain last-modified column could never tell you.


In [ ]:
undone = table(spark, "gym_tracker", "challenge_completions",
               include_deleted=True).where('_deleted_at IS NOT NULL')
print("completions later undone:", undone.count())
undone.select("item_id", "day", "_actor", "_changed_at").orderBy("day").show(20, False)


## Weight against the goal


In [ ]:
wl = gym["weight_logs"].withColumn("day", day("ts"))
(wl.select("day", "weight_kg", "body_fat_pct", "device")
   .orderBy(F.col("day").desc()).toPandas())


## Garmin: sleep, stress, body battery

One row per day, and **missing is not zero** — a day the watch never synced
has nulls, which must stay holes rather than becoming zeros in a mean.


In [ ]:
g = gym["garmin_daily"]
(g.select("day",
          F.round(F.col("sleep_seconds") / 3600, 1).alias("sleep_h"),
          "resting_hr", "stress_avg", "body_battery_high", "body_battery_low")
  .orderBy(F.col("day").desc()).toPandas())


### Does training move the resting heart rate?

A crude look: resting HR on days with a workout versus days without. Not a
controlled comparison — it is a prompt for a better question.


In [ ]:
trained = (w.withColumn("day", day("ts")).select("day").distinct()
            .withColumn("trained", F.lit(True)))

(g.join(trained, "day", "left")
  .withColumn("trained", F.coalesce(F.col("trained"), F.lit(False)))
  .where(F.col("resting_hr").isNotNull())
  .groupBy("trained")
  .agg(F.count("*").alias("days"),
       F.round(F.avg("resting_hr"), 1).alias("resting_hr"),
       F.round(F.avg("sleep_seconds") / 3600, 1).alias("sleep_h"))
  .toPandas())


## Coop: eggs and cost

`logs` holds both egg collections and expenses, told apart by `type`.


In [ ]:
coop = tables(spark, "coop_tracker")
logs = coop["logs"]
logs.groupBy("type").count().orderBy(F.col("count").desc()).show()


In [ ]:
eggs = logs.where(F.col("type") == "eggs").withColumn("day", day("ts"))
(eggs.groupBy("day").agg(F.sum("count").alias("eggs"))
     .orderBy(F.col("day").desc()).toPandas())


### Eggs per bird per week

Only birds currently marked active — a flock that changed size mid-period
would otherwise make the rate meaningless.


In [ ]:
active = coop["chickens"].where(F.col("status") == "active").count()
print("active birds:", active)

(eggs.withColumn("week", F.date_trunc("week", F.to_date("day")))
     .groupBy("week").agg(F.sum("count").alias("eggs"))
     .withColumn("per_bird", F.round(F.col("eggs") / F.lit(max(active, 1)), 2))
     .orderBy(F.col("week").desc()).toPandas())


## Naming the tables — optional

Everything above addresses tables by path, which is what the pipeline does out
of the box. With the **Pipeline Metastore** add-on running *and* the Pipeline
Spark add-on's `metastore_uris` pointing at it, the same tables also get names,
and SQL becomes an option.

`register()` is metadata only — it moves and rewrites nothing, and Delta keeps
the schema in its own log, so a tracker gaining a column needs no re-run. It
creates two names per table: `<name>` as the merge wrote it (payload still a
JSON string) and `<name>_typed`, a view applying the same schema and live-row
filter that `table()` applies in Python.

Without the metastore this cell prints why and changes nothing.


In [ ]:
from lakehouse import catalog_available, register

if catalog_available(spark):
    print(register(spark))
    spark.sql("SHOW DATABASES").show()
else:
    print("no Hive catalog in this session - skipping.\n"
          "Install the Pipeline Metastore add-on and set the Pipeline Spark\n"
          "add-on's metastore_uris to thrift://172.30.32.1:9083.")


In [ ]:
# The typed view answers in SQL what total_reps() answers in Python. Same
# caveat as ever: `sets` is null for single-set entries, so COALESCE it.
if catalog_available(spark):
    spark.sql("""
        SELECT substr(ts, 1, 10) AS day,
               sum(coalesce(sets, 1) * reps) AS reps,
               count(*) AS entries
        FROM gym_tracker.workout_logs_typed
        GROUP BY 1 ORDER BY 1 DESC
    """).show(10)


---

### Where to go next

`lakehouse.SCHEMAS` lists every table with a known schema; `raw()` gives the
untouched JSON for anything not covered, including a column added to a tracker
since this file was written.


In [ ]:
from lakehouse import SCHEMAS
{src: sorted(t) for src, t in SCHEMAS.items()}
